# Explainable ML for Credit Risk Prediction (German Credit + SHAP)
Zaara Riyaz Khan | Minor Project | Run top to bottom (Runtime > Run all).

Pipeline: load data, EDA, preprocessing (winsorize, encode, scale, SMOTE), four tuned models, metrics, SHAP global and local plots, export for Streamlit.

In [ ]:
!pip -q install shap xgboost imbalanced-learn ucimlrepo

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import shap

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 42
os.makedirs("figures", exist_ok=True)
os.makedirs("deploy", exist_ok=True)

COLS = ["status", "duration", "credit_history", "purpose", "credit_amount", "savings", "employment",
        "installment_rate", "personal_status", "other_debtors", "residence_since", "property", "age",
        "other_installment_plans", "housing", "existing_credits", "job", "num_dependents",
        "telephone", "foreign_worker"]
NUM_COLS = ["duration", "credit_amount", "installment_rate", "residence_since", "age",
            "existing_credits", "num_dependents"]
ORD_COLS = ["status", "savings"]
# order used for ordinal encoding (A14 = no checking account, A65 = unknown savings placed last)
ORD_CATS = [["A11", "A12", "A13", "A14"], ["A61", "A62", "A63", "A64", "A65"]]
NOM_COLS = [c for c in COLS if c not in NUM_COLS + ORD_COLS]
print(len(NUM_COLS), "numerical |", len(ORD_COLS), "ordinal |", len(NOM_COLS), "nominal")

## 1. Load the German Credit dataset (UCI)

In [ ]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
try:
    df = pd.read_csv(URL, sep=r"\s+", header=None, names=COLS + ["risk"])
except Exception as e:
    print("Direct download failed, using ucimlrepo instead:", e)
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=144)
    X_ = ds.data.features.copy(); X_.columns = COLS
    df = X_.assign(risk=ds.data.targets.iloc[:, 0].values)

df["risk"] = df["risk"].map({1: 0, 2: 1})   # 0 = good credit, 1 = bad credit (positive class)
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print("Missing values:", int(df.isnull().sum().sum()))
print(df["risk"].value_counts(normalize=True).rename({0: "good", 1: "bad"}).round(3))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(x="risk", data=df, ax=ax[0]); ax[0].set_title("Target distribution (0=good, 1=bad)")
sns.histplot(df["credit_amount"], kde=True, ax=ax[1]); ax[1].set_title("Credit amount (right-skewed)")
sns.heatmap(df[NUM_COLS].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax[2]); ax[2].set_title("Correlation")
plt.tight_layout(); plt.savefig("figures/eda_overview.png", dpi=200); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for a, c in zip(ax, ["credit_amount", "age", "duration"]):
    sns.boxplot(x=df[c], ax=a); a.set_title(f"Outliers: {c}")
plt.tight_layout(); plt.savefig("figures/eda_boxplots.png", dpi=200); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, c in zip(ax, ["status", "credit_history", "purpose"]):
    sns.countplot(x=c, hue="risk", data=df, ax=a); a.set_title(f"{c} vs risk")
plt.tight_layout(); plt.savefig("figures/eda_categorical.png", dpi=200); plt.show()

## 3. Preprocessing
Split first (80/20, stratified), then fit everything on the training set only to avoid leakage: winsorize credit amount and age, one-hot nominal, ordinal-encode checking and savings, scale numerics. SMOTE is applied to the training data only (inside cross-validation during tuning, and for the SHAP/export steps the models are the ones fitted on the SMOTE-balanced data).

In [ ]:
X = df[COLS]; y = df["risk"]
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", X_train_raw.shape, "Test:", X_test_raw.shape)

# winsorisation caps (1st/99th percentile, learned on training data)
CAPS = {c: (float(X_train_raw[c].quantile(0.01)), float(X_train_raw[c].quantile(0.99)))
        for c in ["credit_amount", "age"]}
print("Caps:", CAPS)

def winsorize(d):
    d = d.copy()
    for c, (lo, hi) in CAPS.items():
        d[c] = d[c].clip(lo, hi)
    return d

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM_COLS),
    ("ord", OrdinalEncoder(categories=ORD_CATS), ORD_COLS),
    ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=False), NOM_COLS),
], verbose_feature_names_out=False)

X_train = preprocessor.fit_transform(winsorize(X_train_raw))
X_test = preprocessor.transform(winsorize(X_test_raw))
FEATURES = list(preprocessor.get_feature_names_out())
y_train_arr, y_test_arr = y_train.values, y_test.values
print("Processed shapes:", X_train.shape, X_test.shape, "| features:", len(FEATURES))

# just to show the effect of SMOTE (tuning below applies it inside each CV fold)
_, y_res = SMOTE(random_state=SEED).fit_resample(X_train, y_train_arr)
print("Class counts before SMOTE:", np.bincount(y_train_arr), "after:", np.bincount(y_res))

## 4. Train and tune the four models (GridSearchCV, scoring = F1 on the bad-credit class)

In [ ]:
search_space = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, random_state=SEED),
                            {"model__C": [0.01, 0.1, 1, 10]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=SEED),
                      {"model__max_depth": [3, 4, 5, 6, 8], "model__min_samples_split": [2, 10, 20]}),
    "Random Forest": (RandomForestClassifier(random_state=SEED),
                      {"model__n_estimators": [200, 400], "model__max_depth": [6, 10, None],
                       "model__min_samples_leaf": [1, 2, 4]}),
    "XGBoost": (XGBClassifier(eval_metric="logloss", random_state=SEED),
                {"model__learning_rate": [0.01, 0.05, 0.1], "model__n_estimators": [100, 300],
                 "model__max_depth": [3, 5]}),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

models, best_params = {}, {}
for name, (est, grid) in search_space.items():
    pipe = ImbPipeline([("smote", SMOTE(random_state=SEED)), ("model", est)])
    gs = GridSearchCV(pipe, grid, scoring="f1", cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train_arr)
    models[name] = gs.best_estimator_.named_steps["model"]
    best_params[name] = gs.best_params_
    print(f"{name:20s} CV F1 = {gs.best_score_:.3f}  best = {gs.best_params_}")

## 5. Evaluation and comparison table

In [ ]:
rows, probs = [], {}
for name, m in models.items():
    p = m.predict_proba(X_test)[:, 1]; pred = (p >= 0.5).astype(int)
    probs[name] = p
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test_arr, pred),
                 "Precision": precision_score(y_test_arr, pred),
                 "Recall": recall_score(y_test_arr, pred),
                 "F1-score": f1_score(y_test_arr, pred),
                 "AUC-ROC": roc_auc_score(y_test_arr, p)})
results = pd.DataFrame(rows).set_index("Model").round(3)
results.to_csv("figures/model_comparison.csv")
display(results)
print("Best per metric:\n", results.idxmax())

# confusion matrices
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, m) in zip(axes, models.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test_arr, m.predict(X_test)),
                           display_labels=["Good", "Bad"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.savefig("figures/confusion_matrices.png", dpi=200); plt.show()

# ROC curves
plt.figure(figsize=(6, 5))
for name, p in probs.items():
    fpr, tpr, _ = roc_curve(y_test_arr, p)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test_arr, p):.3f})")
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.legend(); plt.title("ROC curves"); plt.tight_layout()
plt.savefig("figures/roc_curves.png", dpi=200); plt.show()

## 6. SHAP - global explanations (summary plot per model)

In [ ]:
BACKGROUND = shap.sample(X_train, 100, random_state=SEED)

def make_explainer(name, model):
    if name == "Logistic Regression":
        return shap.LinearExplainer(model, BACKGROUND)
    return shap.TreeExplainer(model)

def class1_shap(explainer, X_):
    # SHAP values for the 'bad credit' class, handling different return shapes across models/versions
    sv = explainer.shap_values(X_)
    if isinstance(sv, list):
        sv = sv[1]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, 1]
    ev = np.atleast_1d(explainer.expected_value)
    base = float(ev[1] if len(ev) > 1 else ev[0])
    return sv, base

explainers = {n: make_explainer(n, m) for n, m in models.items()}
shap_vals = {}
for name, ex in explainers.items():
    sv, _ = class1_shap(ex, X_test)
    shap_vals[name] = sv
    plt.figure()
    shap.summary_plot(sv, X_test, feature_names=FEATURES, max_display=15, show=False)
    plt.title(f"SHAP summary - {name}")
    plt.tight_layout()
    plt.savefig(f"figures/shap_summary_{name.replace(' ', '_').lower()}.png", dpi=200, bbox_inches="tight")
    plt.show()

# do the same features rank at the top for every model?
imp = pd.DataFrame({n: np.abs(sv).mean(0) for n, sv in shap_vals.items()}, index=FEATURES)
top = pd.DataFrame({n: imp[n].sort_values(ascending=False).head(10).index.tolist() for n in imp.columns})
top.index = range(1, 11); top.index.name = "rank"
display(top)

## 7. SHAP - local explanations (waterfall) for Logistic Regression

In [ ]:
LR_NAME = "Logistic Regression"
lr = models[LR_NAME]
p_lr = lr.predict_proba(X_test)[:, 1]; pred_lr = (p_lr >= 0.5).astype(int)

idx_correct = int(np.where((y_test_arr == 1) & (pred_lr == 1))[0][0])   # correctly flagged bad credit
idx_wrong = int(np.where(pred_lr != y_test_arr)[0][0])                  # misclassified applicant

sv_lr, base_lr = class1_shap(explainers[LR_NAME], X_test)
for label, i in [("correct", idx_correct), ("misclassified", idx_wrong)]:
    exp = shap.Explanation(values=sv_lr[i], base_values=base_lr, data=X_test[i], feature_names=FEATURES)
    plt.figure()
    shap.plots.waterfall(exp, max_display=12, show=False)
    plt.title(f"{label} (true={y_test_arr[i]}, predicted={pred_lr[i]}, P(bad)={p_lr[i]:.2f})")
    plt.savefig(f"figures/shap_waterfall_{label}.png", dpi=200, bbox_inches="tight")
    plt.show()

## 8. Export for the Streamlit app

In [ ]:
from importlib.metadata import version

artifacts = {
    "preprocessor": preprocessor,
    "models": models,
    "caps": CAPS,
    "columns": COLS,
    "feature_names": FEATURES,
    "background": BACKGROUND,
    "results": results,
}
joblib.dump(artifacts, "deploy/credit_artifacts.pkl", compress=3)

# pin the EXACT versions used here, so the pickles load on Streamlit Cloud
pkgs = ["scikit-learn", "xgboost", "shap", "joblib", "numpy", "pandas", "matplotlib"]
with open("deploy/requirements.txt", "w") as f:
    f.write("streamlit\n")
    for p in pkgs:
        f.write(f"{p}=={version(p)}\n")
print(open("deploy/requirements.txt").read())
print("Saved:", os.listdir("deploy"))

In [ ]:
# Download the deployment files and the figures for your report
import shutil
from google.colab import files
shutil.make_archive("deploy_bundle", "zip", "deploy")
shutil.make_archive("report_figures", "zip", "figures")
files.download("deploy_bundle.zip")
files.download("report_figures.zip")

**Next:** put `app.py` (from the project), `credit_artifacts.pkl` and `requirements.txt` in a GitHub repo, then deploy it on share.streamlit.io.